![image info](https://user-images.githubusercontent.com/91945811/146929832-e78ff1f4-2739-41b1-acd3-019a39b6a42c.png)


# Titanic-Datensatz: Deskriptive Statistik & Korrelation

## Einführung & Motivation

Der Titanic-Datensatz ist einer der bekanntesten und meistgenutzten Datensätze in der Statistik und im maschinellen Lernen. Er enthält Informationen über die Passagiere der Titanic, die 1912 sank, und erlaubt es, spannende Zusammenhänge zwischen Merkmalen wie Alter, Geschlecht, Ticketklasse und Überlebensrate zu analysieren. 

In dieser Übung wirst du grundlegende deskriptive Methoden anwenden, um den Datensatz zu untersuchen. Ziel ist es, zentrale statistische Konzepte wie Lagemaße, Streuung und Korrelationen in der Praxis zu verstehen. Wir beschäftigen uns auch mit Datenqualität und explorativer Datenanalyse.

Am Ende wirst du in der Lage sein:
- Einen Überblick über den Datensatz zu gewinnen,
- Fehlende Werte zu identifizieren und zu behandeln,
- Statistische Maßzahlen zu berechnen,
- Gruppierte Analysen durchzuführen,
- Korrelationen zu interpretieren.

## Vorbereitung: Titanic-Datensatz laden

In [3]:
### Laden über `seaborn`
import pandas as pd
import seaborn as sns

titanic = sns.load_dataset("titanic")
display(titanic)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,deck,embark_town,alive,alone
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,NaN,Southampton,no,False
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,C,Cherbourg,yes,False
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,NaN,Southampton,yes,True
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,C,Southampton,yes,False
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,NaN,Southampton,no,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,NaN,Southampton,no,True
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,B,Southampton,yes,True
888,0,3,female,NaN,1,2,23.4500,S,Third,woman,False,NaN,Southampton,no,False
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,C,Cherbourg,yes,True



---

## 1. Grundlegender Datenüberblick

### Aufgabe: Erste Inspektion
- Nutze `describe()`, `shape`, und `info()`, um die Struktur des Datensatzes zu untersuchen.

**Musterlösung:**

In [5]:
# Ausgabe der Form des Datensatzes (Anzahl der Zeilen und Spalten)
print(titanic.shape)

(891, 15)


In [6]:
# Überblick über Spaltennamen, Datentypen und fehlende Werte
print(titanic.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 15 columns):
 #   Column       Non-Null Count  Dtype   
---  ------       --------------  -----   
 0   survived     891 non-null    int64   
 1   pclass       891 non-null    int64   
 2   sex          891 non-null    object  
 3   age          714 non-null    float64 
 4   sibsp        891 non-null    int64   
 5   parch        891 non-null    int64   
 6   fare         891 non-null    float64 
 7   embarked     889 non-null    object  
 8   class        891 non-null    category
 9   who          891 non-null    object  
 10  adult_male   891 non-null    bool    
 11  deck         203 non-null    category
 12  embark_town  889 non-null    object  
 13  alive        891 non-null    object  
 14  alone        891 non-null    bool    
dtypes: bool(2), category(2), float64(2), int64(4), object(5)
memory usage: 80.7+ KB
None


In [7]:
# Anzeige diskriptiven Statistiken
print(titanic.describe())

         survived      pclass         age       sibsp       parch        fare
count  891.000000  891.000000  714.000000  891.000000  891.000000  891.000000
mean     0.383838    2.308642   29.699118    0.523008    0.381594   32.204208
std      0.486592    0.836071   14.526497    1.102743    0.806057   49.693429
min      0.000000    1.000000    0.420000    0.000000    0.000000    0.000000
25%      0.000000    2.000000   20.125000    0.000000    0.000000    7.910400
50%      0.000000    3.000000   28.000000    0.000000    0.000000   14.454200
75%      1.000000    3.000000   38.000000    1.000000    0.000000   31.000000
max      1.000000    3.000000   80.000000    8.000000    6.000000  512.329200


In [8]:
# Anzeige diskriptiven Statistiken
titanic.describe(include=["object"])

,sex,embarked,who,embark_town,alive
count,891,889,891,889,891
unique,2,3,3,3,2
top,male,S,man,Southampton,no
freq,577,644,537,644,549


**Erklärung:**
Diese ersten Analysen helfen uns, die Struktur des Datensatzes zu verstehen. Wir sehen sofort:
- Welche Spalten enthalten welche Werte,
- Welche Datentypen vorliegen (numerisch, kategorisch),
- Ob es fehlende Werte gibt.

### Aufgabe: Fehlende Werte prüfen
- Verwende `isnull().sum()`, um fehlende Werte (`NaN`) in den Spalten zu analysieren.
- Entscheide, ob du sie entfernen (`dropna()`) oder durch sinnvolle Werte ersetzen möchtest (`fillna()`).

**Musterlösung:**

In [10]:
# Anzeige der Anzahl fehlender Werte je Spalte
print(titanic.isnull().sum())

survived         0
pclass           0
sex              0
age            177
sibsp            0
parch            0
fare             0
embarked         2
class            0
who              0
adult_male       0
deck           688
embark_town      2
alive            0
alone            0
dtype: int64


In [11]:
# Fehlende Alterswerte mit dem Median füllen
# - `.transform()` wird anstelle von `.apply()` verwendet, da es die ursprüngliche Indexstruktur beibehält.
titanic["age"] = titanic.groupby(["sex", "pclass"])["age"].transform(lambda x: x.fillna(x.median()))

# Fehlende Einschiffungshäfen mit dem häufigsten Wert füllen
# - `.mode()` gibt eine Liste der häufigsten Werte zurück, daher wählen wir den ersten mit `[0]`.
titanic["embarked"] = titanic["embarked"].fillna(titanic["embarked"].mode()[0])
titanic["embark_town"] = titanic["embark_town"].fillna(titanic["embark_town"].mode()[0])

# Deck-Spalte entweder entfernen oder mit 'Unknown' füllen
titanic.drop(columns=["deck"], inplace=True)
# Alternative: titanic['deck'] = titanic['deck'].fillna('U')

**Erklärung:**
Fehlende Werte können Analysen beeinflussen. 
- Besonders das Alter (`age`) hat fehlende Werte, weshalb wir den Median als Ersatzwert nutzen. Ältere Passagiere waren oft in höheren Klassen, daher kann ein Median pro Geschlecht und Klasse sinnvoll sein. Alternativ könnten Werte entfernt werden, aber das würde Daten verlieren.
- Kategorische Spalten embarked und embark_town: Beide Spalten hängen zusammen (da embark_town der ausgeschriebene Name von embarked ist). Hier bietet sich das häufigste Vorkommen (mode()) als Ersatz an. Da nur 2 Werte fehlen, ist der häufigste Wert (mode()[0]) eine sinnvolle Wahl.
- Spalte deck hat 688 fehlende Werte: Fast 80% fehlen → Droppen sinnvoll, da kaum verwertbare Daten vorhanden. Falls die Spalte wichtig erscheint, könnte man "U" (Unknown) als Ersatzwert nehmen.

### Aufgabe: Datentypen analysieren
- Prüfe, ob es sinnvoll ist, `pclass` als Kategorie (`category`) zu speichern.
- Überlege, ob `age` oder `fare` in Kategorien (z.B. Altersgruppen) unterteilt werden sollten.

**Musterlösung:**

In [14]:
# Ticketklasse als Kategorie speichern
titanic["pclass"] = titanic["pclass"].astype("category")

# Alter in Gruppen unterteilen
titanic["age_group"] = pd.cut(
    titanic["age"], bins=[0, 18, 40, 60, 100], labels=["Kind", "Jung", "Mittel", "Alt"]
)

In [15]:
display(titanic)

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone,age_group
0,0,3,male,22.0,1,0,7.2500,S,Third,man,True,Southampton,no,False,Jung
1,1,1,female,38.0,1,0,71.2833,C,First,woman,False,Cherbourg,yes,False,Jung
2,1,3,female,26.0,0,0,7.9250,S,Third,woman,False,Southampton,yes,True,Jung
3,1,1,female,35.0,1,0,53.1000,S,First,woman,False,Southampton,yes,False,Jung
4,0,3,male,35.0,0,0,8.0500,S,Third,man,True,Southampton,no,True,Jung
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
886,0,2,male,27.0,0,0,13.0000,S,Second,man,True,Southampton,no,True,Jung
887,1,1,female,19.0,0,0,30.0000,S,First,woman,False,Southampton,yes,True,Jung
888,0,3,female,21.5,1,2,23.4500,S,Third,woman,False,Southampton,no,False,Jung
889,1,1,male,26.0,0,0,30.0000,C,First,man,True,Cherbourg,yes,True,Jung


**Erklärung:**
Durch die Umwandlung von `pclass` in eine Kategorie können Speicherplatz gespart und Analysen vereinfacht werden. Das Alter wird in Gruppen unterteilt, um es besser auswerten zu können.

---

## 2. Deskriptive Statistik



### Aufgabe: Gruppenvergleiche
- Berechne den Durchschnitt von `age` und `fare` für die Gruppen `sex`.

**Musterlösung:**

In [18]:
print(titanic.groupby("sex")[["age", "fare"]].mean())

              age       fare
sex                         
female  27.261146  44.479818
male    30.119879  25.523893


**Erklärung:**
- Diese Berechnung zeigt, ob es **Unterschiede zwischen Männern und Frauen** bezüglich Altersstruktur und Ticketpreisen gab.

### Aufgabe: Korrelation berechnen
- Berechne die **Pearson-Korrelation** zwischen `age` und `fare`.

**Musterlösung:**

In [20]:
print(titanic[["age", "fare"]].corr())

           age      fare
age   1.000000  0.122692
fare  0.122692  1.000000


**Erklärung:**

Der Pearson-Korrelationskoeffizient ist eine statistische Kennzahl, die den linearen Zusammenhang zwischen zwei numerischen Variablen misst.
Der Pearson-Korrelationskoeffizient zwischen age und fare ist nahe 0, was bedeutet, dass kein signifikanter Zusammenhang zwischen dem Alter und dem Ticketpreis besteht.
Dies deutet darauf hin, dass das Alter einer Person keinen starken Einfluss darauf hatte, wie viel für das Ticket bezahlt wurde. 

---

### Aufgabe: Ausreißer identifizieren
- Bestimme Werte von `fare`, die größer sind als `mean + 3*std()`.

**Musterlösung:**

In [23]:
upper_threshold = titanic["fare"].mean() + 3 * titanic["fare"].std()
display(titanic[titanic["fare"] > upper_threshold])

,survived,pclass,sex,age,sibsp,parch,fare,embarked,class,who,adult_male,embark_town,alive,alone,age_group
27,0,1,male,19.0,3,2,263.0000,S,First,man,True,Southampton,no,False,Jung
88,1,1,female,23.0,3,2,263.0000,S,First,woman,False,Southampton,yes,False,Jung
118,0,1,male,24.0,0,1,247.5208,C,First,man,True,Cherbourg,no,False,Jung
258,1,1,female,35.0,0,0,512.3292,C,First,woman,False,Cherbourg,yes,True,Jung
299,1,1,female,50.0,0,1,247.5208,C,First,woman,False,Cherbourg,yes,False,Mittel
311,1,1,female,18.0,2,2,262.3750,C,First,woman,False,Cherbourg,yes,False,Kind
341,1,1,female,24.0,3,2,263.0000,S,First,woman,False,Southampton,yes,False,Jung
377,0,1,male,27.0,0,2,211.5000,C,First,man,True,Cherbourg,no,False,Jung
380,1,1,female,42.0,0,0,227.5250,C,First,woman,False,Cherbourg,yes,True,Mittel
438,0,1,male,64.0,1,4,263.0000,S,First,man,True,Southampton,no,False,Alt


### Aufgabe 8: Min, Max und Quartile analysieren
- Bestimme `min()` und `max()` für `fare`.
- Bestimme den **Interquartilsabstand (IQR = 75 % - 25 %)**.

**Musterlösung:**

In [25]:
print(titanic["fare"].min(), titanic["fare"].max())
iqr = titanic["fare"].quantile(0.75) - titanic["fare"].quantile(0.25)
print(iqr)

0.0 512.3292
23.0896


---

## 4. Gruppenunterschiede

### Aufgabe: Vergleich Überlebender vs. Nicht-Überlebender
- Berechne den Durchschnitt von `age` und `fare` für `survived == 1` und `survived == 0`.

**Musterlösung:**

In [27]:
print(titanic.groupby("survived")[["age", "fare"]].mean())

                age       fare
survived                      
0         29.737705  22.117887
1         28.108684  48.395408


### Aufgabe: Ticketklasse untersuchen
- Berechne den Durchschnitt von `age` und `fare` pro `pclass`.

**Musterlösung:**

In [29]:
print(titanic.groupby("pclass", observed=False)[["age", "fare"]].mean())
# observed=False → Zeigt auch nicht vorhandene Kategorien (falls pclass z. B. eine leere Klasse hätte).

              age       fare
pclass                      
1       38.270463  84.154687
2       29.863207  20.662183
3       24.802281  13.675550


---

## Fazit

Diese Übung hilft dir, zentrale Methoden der deskriptiven Statistik anzuwenden und die Struktur realer Datensätze zu verstehen. Mit weiteren Analysen könnten **Machine Learning Modelle** trainiert oder **Vorhersagen über das Überleben** getroffen werden.

**Viel Erfolg beim Analysieren!**